In [ ]:
# This notebook downloads monthly Divvy bike trip data for July 2025 through June 2026,
# merges all monthly files into one unified DataFrame, removes trailing metadata columns,
# and writes the cleaned dataset to a CSV file.
import io
import zipfile
import pandas as pd
import requests


This notebook downloads Divvy bike trip data from July 2025 through June 2026, merges the monthly files into a single dataset, removes trailing metadata columns, and saves the cleaned result as a CSV file.


In [ ]:
# Build a list of Divvy trip data URLs covering July 2025 through June 2026.
# The source file names are formatted as YYYYMM-divvy-tripdata.zip.
urls = []
for year in range(2025, 2027):
    start = 7 if year == 2025 else 1
    end = 13 if year == 2025 else 7
    for i in range(start, end):
        num = f"{i:02d}"
        urls.append(f"https://divvy-tripdata.s3.amazonaws.com/{year}{num}-divvy-tripdata.zip")
# The `urls` list now contains 12 monthly zip file locations that the notebook will download.


In [ ]:
# Download each monthly zip archive, extract the CSV file inside, and load it into a DataFrame.
# All monthly DataFrames are collected and then concatenated into a single dataset.
monthly_dfs = []
for url in urls:
    response = requests.get(url)
    response.raise_for_status()
    zip_bytes = io.BytesIO(response.content)
    with zipfile.ZipFile(zip_bytes) as z:
        for filename in z.namelist():
            if filename.endswith('.csv'):
                with z.open(filename) as f:
                    monthly_dfs.append(pd.read_csv(f, encoding='latin-1'))
df = pd.concat(monthly_dfs, ignore_index=True)
print(f"Finish! Total rows concatenated: {len(df)}")


Finish! Total rows concated: 5932350


In [ ]:
# Remove the last three columns from the dataset.
# Some Divvy export files include trailing metadata or extra fields that are not needed
# in the final merged dataset.
columns_to_drop = df.columns[-3:].tolist()
df_final = df.drop(columns=columns_to_drop)


In [ ]:
# Save the merged raw trip data to a CSV file in the current working directory.
df_final.to_csv("bike_trip_data(July 2025-June 2026).csv", index=False)
